# Part 1: Introduction and Setup
## Agentic Security Hands-On Lab

**Estimated Time:** 30 minutes

---

## 🎯 Learning Objectives

By the end of this lab series, you will understand:

1. **OAuth 2.0 Token Exchange (RFC 8693)** - Converting user tokens to agent-specific tokens
2. **Policy-Based Access Control** - Fine-grained permissions enforcement with Vault
3. **External Group Mapping** - Mapping IBM Verify groups to Vault policies
4. **Dynamic Secrets** - Short-lived database credentials generated on-demand
5. **Modular Agent Architecture** - Framework-agnostic AI agent design

---

## 📚 Lab Scenario

You are building a secure AI chatbot for an HR department. The chatbot needs to:

- Query employee information from a PostgreSQL database
- Respect user permissions (basic HR staff vs. HR admins)
- Use dynamic, short-lived database credentials
- Maintain a complete audit trail

### Two User Personas:

| User | Group | Permissions |
|------|-------|-------------|
| **Alice** | hr-basic | Can view basic employee info (name, department, email) |
| **Bob** | hr-admin | Can view both basic info AND salary information |

---

## 💡 Core Concept: OAuth 2.0 Token Exchange

### What is Token Exchange (RFC 8693)?

Token exchange allows you to convert one type of token into another:

```
User Token (from IBM Verify) → Agent Token (with specific permissions)
```

### Why is this important for AI Agents?

1. **Principle of Least Privilege** - Agent only gets needed permissions
2. **User Context Preservation** - Agent acts on behalf of the user
3. **Audit Trail** - All actions traceable to original user
4. **Token Scoping** - Different tokens for different purposes

### The Complete Flow:

```
┌─────────┐      ┌──────────────┐      ┌─────────┐      ┌──────────┐
│  User   │─────▶│  IBM Verify  │─────▶│  Agent  │─────▶│  Vault   │
│ (Alice) │      │  (User JWT)  │      │ (Token  │      │ (Dynamic │
└─────────┘      └──────────────┘      │Exchange)│      │  Creds)  │
                                        └─────────┘      └──────────┘
                                             │                │
                                             ▼                ▼
                                        ┌─────────────────────────┐
                                        │   PostgreSQL Database   │
                                        │  (Employee Information) │
                                        └─────────────────────────┘
```

### Key Security Principles:

- **Identity Propagation** - User identity flows through entire chain
- **Policy Enforcement** - Vault checks user groups before granting access
- **Dynamic Credentials** - Database passwords generated on-demand and expire
- **Zero Trust** - Every request is authenticated and authorized

---

## 🏗️ Architecture Overview

### System Components

```
┌─────────────────────────────────────────────────────────────────┐
│                   Agent Framework Layer                         │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐      │
│  │Langflow  │  │ CrewAI   │  │LangGraph │  │LangChain │      │
│  │ Adapter  │  │ Adapter  │  │ Adapter  │  │ Adapter  │      │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘  └────┬─────┘      │
│       └─────────────┴─────────────┴─────────────┘             │
│                   ┌─────────▼─────────┐                        │
│                   │  Base Adapter     │                        │
│                   │  (Abstract Class) │                        │
│                   └─────────┬─────────┘                        │
└─────────────────────────────┼─────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                      Security Layer                             │
│  ┌──────────────────┐         ┌──────────────────┐            │
│  │ Token Exchange   │────────▶│  Vault Auth      │            │
│  │ (RFC 8693)       │         │  (JWT Auth)      │            │
│  └──────────────────┘         └────────┬─────────┘            │
└─────────────────────────────────────────┼─────────────────────┘
                                          │
                                          ▼
┌─────────────────────────────────────────────────────────────────┐
│                    HashiCorp Vault                              │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐        │
│  │ JWT Auth     │  │ External     │  │ Database     │        │
│  │ Method       │  │ Groups       │  │ Secrets      │        │
│  └──────────────┘  └──────────────┘  └──────┬───────┘        │
└─────────────────────────────────────────────┼─────────────────┘
                                              │
                                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    PostgreSQL Database                          │
│  ┌──────────────────────┐  ┌──────────────────────┐           │
│  │ employee_basic_info  │  │ employee_salary_info │           │
│  │ (name, dept, email)  │  │ (salary, bonus)      │           │
│  └──────────────────────┘  └──────────────────────┘           │
└─────────────────────────────────────────────────────────────────┘
```

### Modular Design Benefits:

1. **Framework Agnostic** - Swap between Langflow, CrewAI, LangGraph, LangChain
2. **Reusable Security** - Token exchange and Vault auth work with any framework
3. **Easy Testing** - Test security independently from agent logic
4. **Future Proof** - Add new frameworks without changing security code

---

## ✅ Prerequisites Check

Before starting, ensure you have:

- ✅ Podman installed and running
- ✅ Podman Compose installed
- ✅ Python 3.9 or higher
- ✅ Configured `.env` file in the lab root directory
- ✅ IBM Verify credentials (or mock credentials for testing)

Let's verify your environment is ready:

## 📦 Setting Up Your Environment (VS Code)

### Step 1: Create Virtual Environment

Open a terminal in VS Code and run:

> **⚠️ Important:** Make sure that your VS Code root workspace is set to `agentic-security-incubation` directory before running these commands.

```bash
cd agentic-security-incubation
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
pip install ipykernel
python -m ipykernel install --user --name=.venv --display-name "Python (.venv)"
```

### Step 2: Select Python Interpreter

1. Press `⇧⌘P` (Mac) or `Ctrl+Shift+P` (Windows/Linux) to open Command Palette
2. Type and select: **Python: Select Interpreter**
3. Choose: `./venv/bin/python` (or `./venv/Scripts/python.exe` on Windows)


### ✅ You're Ready!

Once dependencies are installed, you can run all notebook cells. VS Code will:
- Use the venv Python interpreter
- Load environment variables from `.env` file
- Provide IntelliSense for all installed packages

In [2]:
import sys
import subprocess

print("📦 Installing required dependencies...\n")
print("This may take a few minutes on first run.\n")

# Install from requirements.txt
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All dependencies installed successfully!")
    print("\n📚 Installed packages include:")
    print("  • python-dotenv - Environment variable management")
    print("  • hvac - HashiCorp Vault client")
    print("  • psycopg2-binary - PostgreSQL database adapter")
    print("  • pandas - Data analysis and manipulation")
    print("  • requests - HTTP library")
    print("  • PyJWT - JSON Web Token handling")
    print("  • and more...")
else:
    print("❌ Error installing dependencies:")
    print(result.stderr)
    print("\n💡 Try running manually: pip install -r ../requirements.txt")

📦 Installing required dependencies...

This may take a few minutes on first run.

✅ All dependencies installed successfully!

📚 Installed packages include:
  • python-dotenv - Environment variable management
  • hvac - HashiCorp Vault client
  • psycopg2-binary - PostgreSQL database adapter
  • pandas - Data analysis and manipulation
  • requests - HTTP library
  • PyJWT - JSON Web Token handling
  • and more...


In [2]:
import os
import sys
import subprocess
from pathlib import Path

# Add parent directory to path
lab_root = Path.cwd().parent
sys.path.insert(0, str(lab_root))

print(f"📁 Lab Root Directory: {lab_root}")
print(f"🐍 Python Version: {sys.version}")
print(f"📍 Current Directory: {Path.cwd()}")

# Check Podman installation
print("\n🔍 Checking Podman installation...")
try:
    podman_version = subprocess.run(['podman', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Podman: {podman_version.stdout.strip()}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌ Podman not found. Please install Podman.")
    print("   Visit: https://podman.io/getting-started/installation")

# Check Podman Compose installation
try:
    compose_version = subprocess.run(['podman-compose', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Podman Compose: {compose_version.stdout.strip()}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌ Podman Compose not found. Please install podman-compose.")
    print("   Install with: pip install podman-compose")

📁 Lab Root Directory: /Users/stevendirjayanto/initiatives/agentic-security/agentic-security-incubation
🐍 Python Version: 3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.4.4.1)]
📍 Current Directory: /Users/stevendirjayanto/initiatives/agentic-security/agentic-security-incubation/notebooks

🔍 Checking Podman installation...
✅ Podman: podman version 5.7.0
✅ Podman Compose: podman-compose version 1.5.0
podman version 5.7.0


### Load and Verify Environment Variables

In [3]:
from dotenv import load_dotenv

# Load environment variables
env_file = lab_root / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print(f"✅ Loaded environment from: {env_file}")
else:
    print(f"⚠️  .env file not found at: {env_file}")
    print("Please copy .env.template to .env and configure it.")

✅ Loaded environment from: /Users/stevendirjayanto/initiatives/agentic-security/agentic-security-incubation/.env



---

## 📋 Required Environment Variables

Before proceeding, ensure you have updated the following environment variables in your `.env` file:

### 🔐 IBM Verify Configuration
```
IBM_VERIFY_TENANT
IBM_VERIFY_OIDC_URL
IBM_VERIFY_FRONTEND_CLIENT_ID
IBM_VERIFY_FRONTEND_CLIENT_SECRET
IBM_VERIFY_AGENT_CLIENT_ID
IBM_VERIFY_AGENT_CLIENT_SECRET
IBM_VERIFY_TOOL_CLIENT_ID
IBM_VERIFY_TOOL_CLIENT_SECRET
```
📖 **Setup Guide:** See `docs/IBM_VERIFY_THREE_CLIENT_SETUP.md` for detailed instructions

### 🔑 Flask Secret Key
```
FLASK_SECRET_KEY
```
💡 **Generate using:** `python -c "import secrets; print(secrets.token_hex(32))"`  
Or run: `./generate-secret-key.sh`

### 🤖 Langflow Configuration

⚠️ **IMPORTANT:** These values MUST be obtained from the Langflow UI. You cannot guess them!

#### 1️⃣ `REACT_APP_LANGFLOW_API_KEY`
- **How to get:** Langflow UI → Settings → API Keys → Create New Key
- **Example:** `sk-1234567890abcdef...`

#### 2️⃣ `REACT_APP_LANGFLOW_FLOW_ID`
- **How to get:** Langflow UI → Open your flow → Check the URL
- **Example:** `6158c23c-7d05-49db-ba6e-0b3304f7df2a`
- **Location:** The flow ID is in the URL: `http://localhost:7860/flow/<FLOW_ID>`

#### 3️⃣ `REACT_APP_LANGFLOW_COMPONENT_ID`
- **How to get:** Inspect the Vault Credentials Tool component in your flow
- **Example:** `vault_credentials_tool-okuBC`
- **This is critical:** This component receives the JWT token for authentication

**How to find the Component ID:**
1. Open your flow in Langflow UI
2. Click on the **Vault Credentials Tool** component container
3. At the top of the component, click on the **Controls** button (⚙️ icon)
4. You will see the component ID displayed as: `ID: vault_credentials_tool-XXXXX`
5. Copy this ID value (e.g., `vault_credentials_tool-okuBC`)

**Alternative method - Export Flow JSON:**
```bash
# In Langflow UI: Export Flow → Download JSON
# Then search for the component ID:
cat AskHR-Agent.json | grep -A 5 "vault_credentials_tool"
```

---

### ✅ Verification Checklist

Before continuing, verify:
- [ ] All IBM Verify client credentials are configured
- [ ] Flask secret key is generated and set
- [ ] Langflow API key is obtained from Langflow UI
- [ ] Langflow flow ID matches your deployed flow
- [ ] Langflow component ID is correctly identified from the flow

⚠️ **Without these values, the demo application will not work!**

---

## 🔐 Generate Flask Secret Key

Flask applications require a secret key for session management and security. Let's generate a secure random key:

In [ ]:
import secrets

print("🔐 Generating Flask Secret Key...\n")

# Generate a secure random secret key
secret_key = secrets.token_hex(32)

print("✅ Generated Flask Secret Key:")
print(f"   {secret_key}\n")

print("📝 Add this to your .env file:")
print(f"   FLASK_SECRET_KEY={secret_key}\n")

print("💡 This key is used for:")
print("   • Session management in Flask applications")
print("   • Signing cookies and tokens")
print("   • CSRF protection")
print("   • Other cryptographic operations\n")

print("⚠️  Important: Keep this key secret and never commit it to version control!")

---

## 🚀 Start Infrastructure Services

Now let's start all required services using Docker Compose:

- **HashiCorp Vault** - Secrets management and policy enforcement
- **PostgreSQL (Employee DB)** - Employee data storage
- **PostgreSQL (Langflow DB)** - Langflow metadata storage
- **Langflow** - AI agent framework (optional for this lab)

In [5]:
import subprocess
import time

print("🚀 Starting infrastructure services...\n")
print("This may take a few minutes on first run (downloading images)\n")

# Change to lab root directory
os.chdir(lab_root)

# Start Podman Compose
result = subprocess.run(
    ['podman-compose', 'up', '-d'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Podman Compose started successfully!\n")
    print(result.stdout)
    
    print("\n⏳ Waiting for services to be healthy (30 seconds)...")
    for i in range(30, 0, -5):
        print(f"   {i} seconds remaining...")
        time.sleep(5)
    
    print("\n✅ Services should be ready!")
else:
    print("❌ Error starting Podman Compose:")
    print(result.stderr)
    print("\nTroubleshooting:")
    print("  1. Ensure Podman is running. If not run podman machine init and podman machine start")
    print("  2. Check if ports 5432, 7007, 8200 are available")
    print("  3. Try: podman-compose down && podman-compose up -d")

🚀 Starting infrastructure services...

This may take a few minutes on first run (downloading images)

✅ Podman Compose started successfully!

agentic-vault
agentic-postgres
langflow-postgres
agentic-askhr-backend
agentic-langflow
agentic-askhr-frontend


⏳ Waiting for services to be healthy (30 seconds)...
   30 seconds remaining...
   25 seconds remaining...
   20 seconds remaining...
   15 seconds remaining...
   10 seconds remaining...
   5 seconds remaining...

✅ Services should be ready!


### Verify Service Status

In [6]:
# Check service status
status_result = subprocess.run(
    ['podman-compose', 'ps'],
    capture_output=True,
    text=True
)

print("📊 Service Status:")
print("=" * 80)
print(status_result.stdout)

print("\n💡 All services should show 'Up' or 'healthy' status")
print("If any service is not running, check logs with: podman-compose logs <service-name>")

📊 Service Status:
CONTAINER ID  IMAGE                                                        COMMAND               CREATED       STATUS                    PORTS                           NAMES
3af1ce071997  docker.io/hashicorp/vault:latest                             server -dev -dev-...  14 hours ago  Up 5 minutes (healthy)    0.0.0.0:8200->8200/tcp          agentic-vault
22b344588b78  docker.io/library/postgres:15-alpine                         postgres              14 hours ago  Up 5 minutes (healthy)    0.0.0.0:5432->5432/tcp          agentic-postgres
6ba3cf8f966a  docker.io/library/postgres:15-alpine                         postgres              14 hours ago  Up 5 minutes (healthy)    0.0.0.0:5433->5432/tcp          langflow-postgres
a22cc209cc04  localhost/agentic-security-incubation_askhr-backend:latest   python app.py         14 hours ago  Up 5 minutes (healthy)    0.0.0.0:5001->5000/tcp          agentic-askhr-backend
649ff23a0d1f  docker.io/langflowai/langflow:latest          

---

## 🔌 Verify PostgreSQL Database

Let's verify that the PostgreSQL database is running and contains our employee data:

In [7]:
import psycopg2
import pandas as pd

# Database connection parameters
db_params = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': os.getenv('DB_PORT', '5432'),
    'database': os.getenv('DB_NAME', 'employee_db'),
    'user': os.getenv('DB_USER', 'postgres'),
    'password': os.getenv('DB_PASSWORD', 'postgres')
}

print("🔌 Connecting to PostgreSQL...\n")
print(f"Host: {db_params['host']}:{db_params['port']}")
print(f"Database: {db_params['database']}\n")

try:
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    
    # Check tables
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
        ORDER BY table_name
    """)
    tables = cursor.fetchall()
    
    print("📋 Available Tables:")
    for table in tables:
        print(f"  - {table[0]}")
    
    # Count records
    print("\n📊 Record Counts:")
    cursor.execute("SELECT COUNT(*) FROM employee_basic_info")
    basic_count = cursor.fetchone()[0]
    print(f"  - employee_basic_info: {basic_count} records")
    
    cursor.execute("SELECT COUNT(*) FROM employee_salary_info")
    salary_count = cursor.fetchone()[0]
    print(f"  - employee_salary_info: {salary_count} records")
    
    # Show sample data
    print("\n👥 Sample Employee Basic Info (first 5 records):")
    print("=" * 80)
    df_basic = pd.read_sql_query(
        "SELECT * FROM employee_basic_info ORDER BY employee_id LIMIT 5",
        conn
    )
    print(df_basic.to_string(index=False))
    
    cursor.close()
    conn.close()
    
    print("\n✅ Database verification complete!")
    print("\n💡 Note: We have two tables with different sensitivity levels:")
    print("   • employee_basic_info: Public information (name, dept, email)")
    print("   • employee_salary_info: Sensitive information (salary, bonus)")
    
except Exception as e:
    print(f"❌ Error connecting to database: {e}")
    print("\nTroubleshooting:")
    print("  1. Ensure PostgreSQL container is running")
    print("  2. Check connection parameters in .env file")
    print("  3. Wait a bit longer for database to initialize")

🔌 Connecting to PostgreSQL...

Host: localhost:5432
Database: employee_db

📋 Available Tables:
  - department_summary
  - employee_basic_info
  - employee_directory
  - employee_salary_info

📊 Record Counts:
  - employee_basic_info: 15 records
  - employee_salary_info: 15 records

👥 Sample Employee Basic Info (first 5 records):
 employee_id first_name last_name                         email      department                job_title  hire_date   office_location  manager_id    phone_number                 created_at                 updated_at
           1      Sarah   Johnson     sarah.johnson@company.com     Engineering Senior Software Engineer 2020-03-15 San Francisco, CA         NaN +1-415-555-0101 2026-02-20 02:14:14.214330 2026-02-20 02:14:14.214330
           2    Michael      Chen      michael.chen@company.com     Engineering        Software Engineer 2021-06-01 San Francisco, CA         1.0 +1-415-555-0102 2026-02-20 02:14:14.214330 2026-02-20 02:14:14.214330
           3      Emil

/var/folders/1g/q60qy61s2h3_mrzwwb9jb5980000gn/T/ipykernel_3605/2034938217.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_basic = pd.read_sql_query(


---

## ✅ Setup Complete!

Congratulations! You have successfully:

- ✅ Verified your environment configuration
- ✅ Started all infrastructure services
- ✅ Verified PostgreSQL database and sample data
- ✅ Understood the core concepts and architecture

### 📝 What We Learned:

1. **OAuth Token Exchange** - Converting user tokens to agent tokens
2. **System Architecture** - Modular design with security layer
3. **Database Structure** - Two tables with different sensitivity levels
4. **Infrastructure Setup** - Podman Compose for local development

### 🎯 Next Steps:

Continue to **Part 2: Vault Configuration** (`02-vault-configuration.ipynb`) where you will:

- Configure HashiCorp Vault
- Set up JWT authentication
- Create policies for access control
- Configure external group mapping

---

**Ready to continue?** Open `02-vault-configuration.ipynb` to proceed!